In [ ]:
import pyarrow as pa
from pathlib import Path
import pandas as pd
import numpy as np


In [ ]:
# Arrow 파일 목록
dataset_path = Path("/Users/given/projects/BioRiskEval_pLM/proteindb/data/sequence_with_tiers_arrow/validation")
arrow_files = sorted(dataset_path.glob("*.arrow"))

print(f"Found {len(arrow_files)} arrow files")

# 첫 파일로 컬럼 확인
with pa.memory_map(str(arrow_files[0]), 'r') as source:
    reader = pa.ipc.open_stream(source)
    sample = reader.read_all().to_pandas()
    print(f"Columns: {sample.columns.tolist()}")


In [ ]:
# 총 row 의 수
total_rows = 0

print("Counting total rows...")
for i, arrow_file in enumerate(arrow_files):
    with pa.memory_map(str(arrow_file), 'r') as source:
        reader = pa.ipc.open_stream(source)
        table = reader.read_all()
    
    total_rows += len(table)
    
    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{len(arrow_files)} files processed, current total: {total_rows:,}")

print(f"\nTotal rows in valid split: {total_rows:,}")


In [ ]:
total_size

In [ ]:
# Tier별 출력 파일 및 통계
tiers = [1, 2, 3, 4, 5, 6]
output_dir = Path("/Users/given/projects/BioRiskEval_pLM/proteindb")

# 각 tier별 파일 핸들 열기
tier_files = {}
tier_counts = {tier: 0 for tier in tiers}
tier_lengths = {tier: [] for tier in tiers}

for tier in tiers:
    tier_files[tier] = open(output_dir / f"tier{tier}_valid_sequences.fasta", 'w')

total_records = 0

# Arrow 파일을 순회하며 처리
for i, arrow_file in enumerate(arrow_files):
    # 파일 읽기
    with pa.memory_map(str(arrow_file), 'r') as source:
        reader = pa.ipc.open_stream(source)
        table = reader.read_all()
    
    # DataFrame으로 변환
    df = table.to_pandas()
    total_records += len(df)
    
    # 각 tier별로 필터링 및 FASTA 쓰기
    for tier in tiers:
        tier_col = f'tier{tier}_included'
        tier_df = df[df[tier_col] == True]
        tier_counts[tier] += len(tier_df)
        
        # FASTA 쓰기
        for _, row in tier_df.iterrows():
            seq = row['sequence']
            tier_lengths[tier].append(len(seq))
            
            tier_files[tier].write(f">{row['sequence_id']}\n")
            for j in range(0, len(seq), 60):
                tier_files[tier].write(seq[j:j+60] + '\n')
    
    # 진행상황 출력
    if (i + 1) % 10 == 0:
        print(f"  {i + 1}/{len(arrow_files)} files processed")

# 파일 닫기
for tier in tiers:
    tier_files[tier].close()

print(f"\n✓ Complete!")
print(f"Total records: {total_records:,}\n")


In [ ]:
# 각 Tier별 통계 출력
for tier in tiers:
    output_path = output_dir / f"tier{tier}_sequences.fasta"
    file_size = output_path.stat().st_size / (1024**2)
    
    print(f"=== Tier {tier} ===")
    print(f"  Records: {tier_counts[tier]:,}")
    print(f"  File: {output_path.name}")
    print(f"  Size: {file_size:.2f} MB")
    
    if tier_lengths[tier]:
        lengths = tier_lengths[tier]
        print(f"  Sequence length:")
        print(f"    Min: {min(lengths)}")
        print(f"    Max: {max(lengths)}")
        print(f"    Mean: {np.mean(lengths):.2f}")
        print(f"    Median: {np.median(lengths):.2f}")
    print()


In [ ]:
# 각 Tier별 길이 분포 히스토그램
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, tier in enumerate(tiers):
    if tier_lengths[tier]:
        lengths = tier_lengths[tier]
        
        axes[i].hist(lengths, bins=50, edgecolor='black', alpha=0.7)
        axes[i].set_xlabel('Sequence Length')
        axes[i].set_ylabel('Frequency')
        axes[i].set_title(f'Tier {tier} - {len(lengths):,} sequences')
        axes[i].grid(True, alpha=0.3)
        
        # 통계 정보 추가
        mean_len = np.mean(lengths)
        median_len = np.median(lengths)
        axes[i].axvline(mean_len, color='r', linestyle='--', linewidth=2, label=f'Mean: {mean_len:.0f}')
        axes[i].axvline(median_len, color='g', linestyle='--', linewidth=2, label=f'Median: {median_len:.0f}')
        axes[i].legend()

plt.tight_layout()
plt.savefig('/Users/given/projects/BioRiskEval_pLM/proteindb/tier_length_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

print("Histogram saved: tier_length_distributions.png")


In [ ]:
# ESM2 토크나이저 로드
from transformers import AutoTokenizer

print("Loading ESM2 tokenizer...")
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t6_8M_UR50D")
print("Tokenizer loaded!")


In [ ]:
# 각 tier별로 토큰 길이 계산 (샘플링해서 빠르게)
print("Calculating token lengths for each tier...\n")

tier_token_lengths = {}

for tier in tiers:
    if tier_lengths[tier]:
        print(f"Processing Tier {tier}...")
        
        # 토큰 길이 계산을 위해 다시 arrow 파일 순회
        token_lengths = []
        
        for i, arrow_file in enumerate(arrow_files):
            with pa.memory_map(str(arrow_file), 'r') as source:
                reader = pa.ipc.open_stream(source)
                table = reader.read_all()
            
            df = table.to_pandas()
            tier_col = f'tier{tier}_included'
            tier_df = df[df[tier_col] == True]
            
            # 토큰 길이 계산
            for _, row in tier_df.iterrows():
                seq = row['sequence']
                tokens = tokenizer(seq, add_special_tokens=True, return_tensors=None)
                token_lengths.append(len(tokens['input_ids']))
            
            if (i + 1) % 10 == 0:
                print(f"  {i + 1}/{len(arrow_files)} files processed")
        
        tier_token_lengths[tier] = token_lengths
        print(f"  Tier {tier}: {len(token_lengths):,} sequences processed\n")

print("✓ Token length calculation complete!")


In [ ]:
# 토큰 길이가 1024를 넘는 시퀀스의 비율
print("=== Sequences with token length > 1024 ===\n")

for tier in tiers:
    if tier in tier_token_lengths and tier_token_lengths[tier]:
        token_lens = tier_token_lengths[tier]
        total = len(token_lens)
        over_1024 = sum(1 for l in token_lens if l > 1024)
        percentage = (over_1024 / total) * 100
        
        print(f"Tier {tier}:")
        print(f"  Total sequences: {total:,}")
        print(f"  Token length > 1024: {over_1024:,} ({percentage:.2f}%)")
        print(f"  Token length ≤ 1024: {total - over_1024:,} ({100 - percentage:.2f}%)")
        print(f"  Min token length: {min(token_lens)}")
        print(f"  Max token length: {max(token_lens)}")
        print(f"  Mean token length: {np.mean(token_lens):.2f}")
        print(f"  Median token length: {np.median(token_lens):.2f}")
        print()


=== Tier 1 ===
  Records: 2,776
  File: tier1_sequences.fasta
  Size: 0.70 MB
  Sequence length:
    Min: 16
    Max: 3174
    Mean: 241.34
    Median: 156.00

=== Tier 2 ===
  Records: 1,046
  File: tier2_sequences.fasta
  Size: 0.33 MB
  Sequence length:
    Min: 16
    Max: 3170
    Mean: 301.00
    Median: 232.00

=== Tier 3 ===
  Records: 666
  File: tier3_sequences.fasta
  Size: 0.20 MB
  Sequence length:
    Min: 16
    Max: 3170
    Mean: 292.30
    Median: 248.00

=== Tier 4 ===
  Records: 397
  File: tier4_sequences.fasta
  Size: 0.11 MB
  Sequence length:
    Min: 16
    Max: 2238
    Mean: 267.02
    Median: 298.00

=== Tier 5 ===
  Records: 3
  File: tier5_sequences.fasta
  Size: 0.00 MB
  Sequence length:
    Min: 87
    Max: 262
    Mean: 183.67
    Median: 202.00

=== Tier 6 ===
  Records: 1
  File: tier6_sequences.fasta
  Size: 0.00 MB
  Sequence length:
    Min: 87
    Max: 87
    Mean: 87.00
    Median: 87.00


=== Tier 1 ===
  Records: 478,566
  File: tier1_sequences.fasta
  Size: 118.75 MB
  Sequence length:
    Min: 12
    Max: 13556
    Mean: 235.45
    Median: 137.00

=== Tier 2 ===
  Records: 131,371
  File: tier2_sequences.fasta
  Size: 42.45 MB
  Sequence length:
    Min: 12
    Max: 13556
    Mean: 313.39
    Median: 191.00

=== Tier 3 ===
  Records: 58,809
  File: tier3_sequences.fasta
  Size: 17.84 MB
  Sequence length:
    Min: 13
    Max: 7803
    Mean: 293.44
    Median: 170.00

=== Tier 4 ===
  Records: 11,208
  File: tier4_sequences.fasta
  Size: 2.49 MB
  Sequence length:
    Min: 13
    Max: 7096
    Mean: 210.44
    Median: 126.00

=== Tier 5 ===
  Records: 1,173
  File: tier5_sequences.fasta
  Size: 0.55 MB
  Sequence length:
    Min: 13
    Max: 7096
    Mean: 466.67
    Median: 147.00

=== Tier 6 ===
  Records: 167
  File: tier6_sequences.fasta
  Size: 0.07 MB
  Sequence length:
    Min: 22
    Max: 7096
    Mean: 411.45
    Median: 121.00
